In [ ]:

# CNN Training for Exercise Recognition

import numpy as np
import pandas as pd
import zipfile
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

# Unzip dataset
zip_path = "physical+therapy+exercises+dataset.zip"
extract_path = "pt_exercises_dataset"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Load and preprocess data
def load_data(subjects=['s1', 's2', 's3', 's4', 's5'], exercises=['e6', 'e7', 'e8'], unit='u2'):
    data, labels = [], []
    for subject in subjects:
        for e_idx, exercise in enumerate(exercises):
            path = f"{extract_path}/{subject}/{exercise}/{unit}/test.txt"
            if os.path.exists(path):
                df = pd.read_csv(path, sep="\t", header=None)
                header = df.iloc[0, 0].split(";")
                df = df.iloc[1:]
                df = df[0].str.split(";", expand=True)
                df.columns = header
                df = df.apply(pd.to_numeric)
                for start in range(0, len(df)-50+1, 50):
                    window = df.iloc[start:start+50]
                    features = window[['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z']].values
                    mag = np.linalg.norm(window[['acc_x', 'acc_y', 'acc_z']].values, axis=1).reshape(-1, 1)
                    features = np.hstack([features, mag])
                    data.append(features)
                    labels.append(e_idx)
    return np.array(data), np.array(labels)

# Load dataset
X, y = load_data()
y_cat = to_categorical(y, num_classes=3)

# Split into training and validation
X_train, X_val, y_train, y_val = train_test_split(X, y_cat, test_size=0.2, random_state=42, stratify=y)

# Build CNN model
model = Sequential([
    Conv1D(128, 5, activation='relu', input_shape=(50, 7)),
    MaxPooling1D(2),
    Dropout(0.3),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])
model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=64, verbose=1)

# Plot training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label="Training Loss")
plt.plot(history.history['val_loss'], label="Validation Loss")
plt.title("CNN Training and Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()
